# Rainfall Feature Engineering

### This notebook converts daily NASA POWER precipitation data into crop-year rainfall indicators for use in the Crop Yield Risk & Advisory Dashboard.

In [1]:
import pandas as pd
import numpy as np

In [2]:
rainfall_daily = pd.read_csv(
    "../../data/processed/rainfall_daily_2021_2025.csv"
)

In [3]:
rainfall_daily.head()

,Date,Rainfall_mm,Year,State,District,Latitude,Longitude
0,2021-01-01,0.00,2021,Haryana,Ambala,30.3435,76.949355
1,2021-01-02,2.74,2021,Haryana,Ambala,30.3435,76.949355
2,2021-01-03,1.25,2021,Haryana,Ambala,30.3435,76.949355
3,2021-01-04,0.76,2021,Haryana,Ambala,30.3435,76.949355
4,2021-01-05,7.67,2021,Haryana,Ambala,30.3435,76.949355


In [4]:
rainfall_daily["Date"] = pd.to_datetime(
    rainfall_daily["Date"]
)

In [5]:
rainfall_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196678 entries, 0 to 196677
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Date         196678 non-null  datetime64[ns]
 1   Rainfall_mm  196678 non-null  float64       
 2   Year         196678 non-null  int64         
 3   State        196678 non-null  object        
 4   District     196678 non-null  object        
 5   Latitude     196678 non-null  float64       
 6   Longitude    196678 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(2)
memory usage: 10.5+ MB


In [6]:
# We have to allign year with our production data year

def get_crop_year(date):

    if date.month >=7:
        start_year = date.year

    else:
        start_year = date.year-1

    end_year = (start_year+1)%100

    return f"{start_year}-{end_year:02d}"

In [7]:
rainfall_daily["Crop_year"] = (rainfall_daily["Date"].apply(get_crop_year))

In [8]:
rainfall_daily[["Date" , "Crop_year"]].head()

,Date,Crop_year
0,2021-01-01,2020-21
1,2021-01-02,2020-21
2,2021-01-03,2020-21
3,2021-01-04,2020-21
4,2021-01-05,2020-21


In [9]:
# Trying some specific dates 

rainfall_daily[
rainfall_daily["Date"].isin(
    pd.to_datetime([
           "2021-07-01",
            "2021-12-01",
            "2022-01-01",
            "2022-06-30",
            "2022-07-01"
    ])
)
][["Date" , "Crop_year"]]

,Date,Crop_year
181,2021-07-01,2021-22
334,2021-12-01,2021-22
365,2022-01-01,2021-22
545,2022-06-30,2021-22
546,2022-07-01,2022-23
...,...,...
195217,2021-07-01,2021-22
195370,2021-12-01,2021-22
195401,2022-01-01,2021-22
195581,2022-06-30,2021-22


In [10]:
# Creating month 

rainfall_daily["Month"]=(rainfall_daily["Date"].dt.month)

In [11]:
rainfall_daily[
["Date","Month","Crop_year"]
].head()

,Date,Month,Crop_year
0,2021-01-01,1,2020-21
1,2021-01-02,1,2020-21
2,2021-01-03,1,2020-21
3,2021-01-04,1,2020-21
4,2021-01-05,1,2020-21


In [12]:
# Defining the monsoon month - June, July , August , September

rainfall_daily["Is_Monsoon"]=(
    rainfall_daily["Month"].isin([6,7,8,9])
)

In [13]:
rainfall_daily[
    ["Date", "Month", "Is_Monsoon"]
].head(20)

,Date,Month,Is_Monsoon
0,2021-01-01,1,False
1,2021-01-02,1,False
2,2021-01-03,1,False
3,2021-01-04,1,False
4,2021-01-05,1,False
5,2021-01-06,1,False
6,2021-01-07,1,False
7,2021-01-08,1,False
8,2021-01-09,1,False
9,2021-01-10,1,False


In [14]:
# Creating rabi period rainfall flag ( October-March)

rainfall_daily["Is_Rabi_Period"]=(
    rainfall_daily["Month"].isin([10,11,12,1,2,3])
)

In [15]:
# Creating rainy_day indicator ( rainfall<2.3mm) must not be count as useful rainfall

rainfall_daily["Is_Rainy_Day"]=(
    rainfall_daily["Rainfall_mm"]>=2.5
)

In [16]:
# Heavy_rain indicator

rainfall_daily["Is_Heavy_Rain_Day"] = (
    rainfall_daily["Rainfall_mm"] >= 64.5
)

In [17]:
# Creating crop-year total rainfall

crop_year_rainfall = (
    rainfall_daily
    .groupby(
        [
            "State",
            "District",
            "Crop_year"
        ],
        as_index=False
    )
    .agg(
        Crop_Year_Rainfall_mm=(
            "Rainfall_mm",
            "sum"
        ),
        Rainy_Days=(
            "Is_Rainy_Day",
            "sum"
        ),
        Heavy_Rain_Days=(
            "Is_Heavy_Rain_Day",
            "sum"
        ),
        Days_Available=(
            "Rainfall_mm",
            "count"
        )
    )
)

In [18]:
# Calculating monsoon year rainfall

monsoon_rainfall = (
    rainfall_daily[
        rainfall_daily["Is_Monsoon"]
    ]
    .groupby(
        [
            "State",
            "District",
            "Crop_year"
        ],
        as_index=False
    )["Rainfall_mm"]
    .sum()
    .rename(
        columns={
            "Rainfall_mm":
            "Monsoon_Rainfall_mm"
        }
    )
)

In [19]:
# Calculating rabi-period rainfall

rabi_rainfall = (
    rainfall_daily[
        rainfall_daily["Is_Rabi_Period"]
    ]
    .groupby(
        [
            "State",
            "District",
            "Crop_year"
        ],
        as_index=False
    )["Rainfall_mm"]
    .sum()
    .rename(
        columns={
            "Rainfall_mm":
            "Rabi_Rainfall_mm"
        }
    )
)

In [20]:
# Merging rainfall feature together

rainfall_features = (
    crop_year_rainfall
    .merge(
        monsoon_rainfall,
        on=[
            "State",
            "District",
            "Crop_year"
        ],
        how="left"
    )
    .merge(
        rabi_rainfall,
        on=[
            "State",
            "District",
            "Crop_year"
        ],
        how="left"
    )
)

In [21]:
rainfall_features.head()

,State,District,Crop_year,Crop_Year_Rainfall_mm,Rainy_Days,Heavy_Rain_Days,Days_Available,Monsoon_Rainfall_mm,Rabi_Rainfall_mm
0,Haryana,Ambala,2020-21,306.15,36,0,181,119.08,64.53
1,Haryana,Ambala,2021-22,1297.57,97,0,365,981.67,246.72
2,Haryana,Ambala,2022-23,1567.66,110,1,365,1207.36,181.25
3,Haryana,Ambala,2023-24,1301.31,84,3,366,1120.44,152.26
4,Haryana,Ambala,2024-25,1447.69,111,1,365,1191.42,133.39


In [22]:
# Checking the completeness

rainfall_features[
    "Days_Available"
].value_counts().sort_index()

Days_Available
181    120
184      2
365    358
366    120
Name: count, dtype: int64

In [23]:
# Indentifying the incomplete rows

incomplete_rainfall = rainfall_features[
    rainfall_features["Days_Available"] < 365
]

incomplete_rainfall[
    [
        "State",
        "District",
        "Crop_year",
        "Days_Available"
    ]
].head(20)

,State,District,Crop_year,Days_Available
0,Haryana,Ambala,2020-21,181
5,Haryana,Bhiwani,2020-21,181
10,Haryana,Charkhi Dadri,2020-21,181
15,Haryana,Faridabad,2020-21,181
20,Haryana,Fatehabad,2020-21,181
25,Haryana,Gurugram,2020-21,181
30,Haryana,Hisar,2020-21,181
35,Haryana,Jhajjar,2020-21,181
40,Haryana,Jind,2020-21,181
45,Haryana,Kaithal,2020-21,181


In [24]:
#Keeping only complete project year

valid_crop_years = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25"

]

rainfall_features_complete = (
    rainfall_features[
        rainfall_features["Crop_year"]
        .isin(valid_crop_years)
    ]
    .copy()
)

In [25]:
rainfall_features_complete[
    "Crop_year"
].value_counts()

Crop_year
2021-22    120
2022-23    120
2023-24    120
2024-25    120
Name: count, dtype: int64

In [26]:
# Creating historical average rainfall

rainfall_features_complete = (
    rainfall_features_complete
    .sort_values(
        [
            "State",
            "District",
            "Crop_year"
        ]
    )
    .reset_index(drop=True)
)

In [27]:
rainfall_features_complete[
    "Historical_Avg_Rainfall"
] = (
    rainfall_features_complete
    .groupby(
        ["State", "District"]
    )["Crop_Year_Rainfall_mm"]
    .transform(
        lambda x:
        x.shift(1).expanding().mean()
    )
)

In [28]:
rainfall_features_complete.head()

,State,District,Crop_year,Crop_Year_Rainfall_mm,Rainy_Days,Heavy_Rain_Days,Days_Available,Monsoon_Rainfall_mm,Rabi_Rainfall_mm,Historical_Avg_Rainfall
0,Haryana,Ambala,2021-22,1297.57,97,0,365,981.67,246.72,NaN
1,Haryana,Ambala,2022-23,1567.66,110,1,365,1207.36,181.25,1297.570000
2,Haryana,Ambala,2023-24,1301.31,84,3,366,1120.44,152.26,1432.615000
3,Haryana,Ambala,2024-25,1447.69,111,1,365,1191.42,133.39,1388.846667
4,Haryana,Bhiwani,2021-22,724.11,52,0,365,610.61,89.75,NaN


In [29]:
# Rainfall Deviation

historical_rainfall_safe = (
    rainfall_features_complete[
        "Historical_Avg_Rainfall"
    ]
    .replace(0, np.nan)
)

In [30]:
rainfall_features_complete[
    "Rainfall_Deviation_Pct"
] = (
    (
        rainfall_features_complete[
            "Crop_Year_Rainfall_mm"
        ]
        -
        rainfall_features_complete[
            "Historical_Avg_Rainfall"
        ]
    )
    /
    historical_rainfall_safe
) * 100

In [31]:
# Renaming crop_year to match production 

rainfall_features_complete = (
    rainfall_features_complete
    .rename(
        columns={
            "Crop_year": "Year"
        }
    )
)



In [32]:
rainfall_features_complete.head()

,State,District,Year,Crop_Year_Rainfall_mm,Rainy_Days,Heavy_Rain_Days,Days_Available,Monsoon_Rainfall_mm,Rabi_Rainfall_mm,Historical_Avg_Rainfall,Rainfall_Deviation_Pct
0,Haryana,Ambala,2021-22,1297.57,97,0,365,981.67,246.72,NaN,NaN
1,Haryana,Ambala,2022-23,1567.66,110,1,365,1207.36,181.25,1297.570000,20.815062
2,Haryana,Ambala,2023-24,1301.31,84,3,366,1120.44,152.26,1432.615000,-9.165407
3,Haryana,Ambala,2024-25,1447.69,111,1,365,1191.42,133.39,1388.846667,4.236849
4,Haryana,Bhiwani,2021-22,724.11,52,0,365,610.61,89.75,NaN,NaN


In [33]:
# Saving the rainfall feature table 

rainfall_features_complete.to_csv(
    "../../data/processed/"
    "rainfall_features_2021_2024.csv",
    index=False
)

In [34]:
# Now merging rainfall with production

production = pd.read_csv(
    "../../data/processed/"
    "production_history_2021_2025.csv"
)



In [35]:
production_for_merge = production[
    production["Year"].isin(
        [
            "2021-22",
            "2022-23",
            "2023-24"
        ]
    )
].copy()

In [36]:
agri_data = production_for_merge.merge(
    rainfall_features_complete,
    on=[
        "State",
        "District",
        "Year"
    ],
    how="left"
)

In [37]:
# Validating the merge 

production_for_merge.shape

(1048, 18)

In [38]:
agri_data.shape

(1048, 26)

In [39]:
agri_data[
    "Crop_Year_Rainfall_mm"
].isnull().sum()

np.int64(0)

In [40]:
missing_rainfall_merge = agri_data[
    agri_data[
        "Crop_Year_Rainfall_mm"
    ].isnull()
][
    [
        "State",
        "District",
        "Crop",
        "Year"
    ]
]

missing_rainfall_merge.drop_duplicates()

,State,District,Crop,Year


In [64]:
rainfall_features[
    rainfall_features["Crop_year"] == "2024-25"
][
    [
        "State",
        "District",
        "Crop_year",
        "Days_Available",
        
    ]
].head(20)

,State,District,Crop_year,Days_Available
4,Haryana,Ambala,2024-25,365
9,Haryana,Bhiwani,2024-25,365
14,Haryana,Charkhi Dadri,2024-25,365
19,Haryana,Faridabad,2024-25,365
24,Haryana,Fatehabad,2024-25,365
29,Haryana,Gurugram,2024-25,365
34,Haryana,Hisar,2024-25,365
39,Haryana,Jhajjar,2024-25,365
44,Haryana,Jind,2024-25,365
49,Haryana,Kaithal,2024-25,365


In [65]:
rainfall_features[
    rainfall_features["Crop_year"] == "2024-25"
]["Days_Available"].value_counts()

Days_Available
365    118
184      2
Name: count, dtype: int64

In [66]:
rainfall_features[
    rainfall_features["District"].isin(
        [
            "Shrawasti",
            "Shahid Bhagat Singh Nagar"
        ]
    )
][
    [
        "State",
        "District",
        "Crop_year",
        "Days_Available",
        "Crop_Year_Rainfall_mm",
        
    ]
]

,State,District,Crop_year,Days_Available,Crop_Year_Rainfall_mm
210,Punjab,Shahid Bhagat Singh Nagar,2020-21,181,258.55
211,Punjab,Shahid Bhagat Singh Nagar,2021-22,365,1060.59
212,Punjab,Shahid Bhagat Singh Nagar,2022-23,365,1359.03
213,Punjab,Shahid Bhagat Singh Nagar,2023-24,366,1308.70
214,Punjab,Shahid Bhagat Singh Nagar,2024-25,184,781.01
565,Uttar Pradesh,Shrawasti,2020-21,181,689.26
566,Uttar Pradesh,Shrawasti,2021-22,365,1615.69
567,Uttar Pradesh,Shrawasti,2022-23,365,1294.03
568,Uttar Pradesh,Shrawasti,2023-24,366,1141.14
569,Uttar Pradesh,Shrawasti,2024-25,184,1220.80


In [67]:
rainfall_features_complete = (
    rainfall_features[
        rainfall_features["Crop_year"].isin(
            valid_crop_years
        )
    ]
    .copy()
)

In [68]:
rainfall_features_complete.head()

,State,District,Crop_year,Crop_Year_Rainfall_mm,Rainy_Days,Heavy_Rain_Days,Days_Available,Monsoon_Rainfall_mm,Rabi_Rainfall_mm
1,Haryana,Ambala,2021-22,1297.57,97,0,365,981.67,246.72
2,Haryana,Ambala,2022-23,1567.66,110,1,365,1207.36,181.25
3,Haryana,Ambala,2023-24,1301.31,84,3,366,1120.44,152.26
4,Haryana,Ambala,2024-25,1447.69,111,1,365,1191.42,133.39
6,Haryana,Bhiwani,2021-22,724.11,52,0,365,610.61,89.75


In [69]:
rainfall_features_complete = (
    rainfall_features_complete.rename(
        columns={
            "Crop_Year": "Year"
        }
    )
)

In [70]:
rainfall_features_complete.to_csv(
    "../../data/processed/rainfall_features_2021_2025.csv",
    index=False
)